In [25]:
from torch.utils.data import DataLoader, TensorDataset

# from model         import GenerateModel
from model import ConvAttnPool
from metrics       import eval_model, compare_metric
from evaluation      import all_metrics
import numpy             as np
import random

In [26]:
from functools import partial
import os
import tempfile
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split
from copy import deepcopy

from ray import tune
from ray import train
# from ray.train import Checkpoint, get_checkpoint
from ray.tune.schedulers import ASHAScheduler
import ray.cloudpickle as pickle

In [27]:
def load_data(data_dir="./data"):
    X_train = np.load('/home/drew/FL-with-MIMIC/Replicating Mullenbach/X_train.npy')
    X_test  = np.load('/home/drew/FL-with-MIMIC/Replicating Mullenbach/X_test.npy')

    Y_train = np.load('/home/drew/FL-with-MIMIC/Replicating Mullenbach/Y_train.npy')
    Y_test  = np.load('/home/drew/FL-with-MIMIC/Replicating Mullenbach/Y_test.npy')

    X_train = torch.from_numpy(X_train).long()
    Y_train = torch.from_numpy(Y_train).type(torch.float32)

    X_test  = torch.from_numpy(X_test) .long()
    Y_test  = torch.from_numpy(Y_test) .type(torch.float32)
    trainset = TensorDataset(X_train,Y_train)
    testset  = TensorDataset(X_test ,Y_test )
    return trainset, testset

In [28]:
def load_client_data(client_idx: int, num_clients: int, idxs, train_subet,limit) -> TensorDataset:
    # X = np.load('/home/drew/FL-with-MIMIC/Replicating Mullenbach/X_train.npy')
    # Y = np.load('/home/drew/FL-with-MIMIC/Replicating Mullenbach/Y_train.npy')

    # X = torch.from_numpy(X).long()
    # Y = torch.from_numpy(Y).type(torch.float32)

    indices = [idx for idx, val in enumerate(idxs) if client_idx == val]
    #X_client = train_subet[indices[:limit-1]]
    #y_client = train_subet[indices[:limit-1]]

    return train_subet[indices[:limit-1]]#TensorDataset(X_client, y_client)

def client_update(model  : nn.Module, train_loader: DataLoader, epochs : int = 1,lr: float = 0.0001, device : str = 'cpu') -> dict:
    model.to(device)
    model.train()
    # optimizer = optim.SGD(model.parameters(), lr=lr)
    optimizer   = torch.optim.Adam(model.parameters(), lr = 0.001, betas =  (0.9,0.99)) #(0.1,0.3)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds, alpha = model(X_batch)
            loss = loss_fn(preds, y_batch)
            loss.backward()
            optimizer.step()

    return deepcopy(model.state_dict())

def server_aggregate(global_model: nn.Module, client_state_dicts: list) -> nn.Module:
    global_dict = global_model.state_dict()
    for key in global_dict.keys():
        stacked = torch.stack([client_dict[key].float() for client_dict in client_state_dicts], dim=0)
        global_dict[key] = torch.mean(stacked, dim=0)
    global_model.load_state_dict(global_dict)
    return global_model

In [29]:
trainset, testset = load_data()

test_abs = int(len(trainset) * 0.8)
train_subset, val_subset = random_split(
    trainset, [test_abs, len(trainset) - test_abs]
)

In [ ]:
def train_model(config, data_dir=None,data_points=None):
    net = ConvAttnPool(num_of_filters = config["l1"], 
                        kernel_size    = config["l2"])

    device = "cpu"
    if torch.cuda.is_available():
        device = torch.device('cuda')
    net.to(device)
    ##########################################
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(net.parameters(), lr=config["lr"])
    ##########################################

    checkpoint = tune.get_checkpoint()
    if checkpoint:
        with checkpoint.as_directory() as checkpoint_dir:
            data_path = Path(checkpoint_dir) / "data.pkl"
            with open(data_path, "rb") as fp:
                checkpoint_state = pickle.load(fp)
            start_epoch = checkpoint_state["epoch"]
            net.load_state_dict(checkpoint_state["net_state_dict"])
            optimizer.load_state_dict(checkpoint_state["optimizer_state_dict"])
    else:
        start_epoch = 0
    ##########################################

    trainset, testset = load_data(data_dir)

    test_abs = int(len(trainset) * 0.8)
    train_subset, val_subset = random_split(
        trainset, [test_abs, len(trainset) - test_abs]
    )

    # trainloader = torch.utils.data.DataLoader(
    #     train_subset, batch_size=int(config["batch_size"]), shuffle=True, num_workers=8
    # )

    valloader = torch.utils.data.DataLoader(
        val_subset, batch_size=int(config["batch_size"]), shuffle=True, num_workers=8
    )


    ##########################################
    c1,c2,c3 = random_split(train_subset,[0.33,0.33,0.34])
    client_loaders = [torch.utils.data.DataLoader(c1, batch_size=int(config["batch_size"]), shuffle=True),
                      torch.utils.data.DataLoader(c2, batch_size=int(config["batch_size"]), shuffle=True),
                      torch.utils.data.DataLoader(c3, batch_size=int(config["batch_size"]), shuffle=True),                  
                     ]
    # client_loaders = []
    # for client_idx in range(3):
    #     dataset = load_client_data(client_idx, 3, idxs=data_points,train_subet = train_subset,limit=test_abs)
    #     loader  = DataLoader(dataset, batch_size=config["batch_size"], shuffle=True)
    #     client_loaders.append(loader)

    for epoch in range(5):
        client_state_dicts = []
        for client_idx, loader in enumerate(client_loaders, 1):
            client_model = ConvAttnPool(num_of_filters = config["l1"], kernel_size = config["l2"])
            client_model.load_state_dict(net.state_dict())
            updated_state = client_update( client_model,loader,epochs=10,device = device)
            client_state_dicts.append(updated_state)
        net = server_aggregate(net, client_state_dicts)
    # for epoch in range(start_epoch, 100):
    #     running_loss = 0.0
    #     epoch_steps = 0
    #     for i, data in enumerate(trainloader, 0):
    #         # get the inputs; data is a list of [inputs, labels]
    #         inputs, labels = data
    #         inputs, labels = inputs.to(device), labels.to(device)

    #         preds, alpha = net(inputs)
    #         preds = preds.type(torch.float32).to(device)
            
    #         loss  = criterion(preds, labels)

    #         # Zero the gradients
    #         optimizer.zero_grad()
    #         loss     .backward()
    #         optimizer.step()

    #         running_loss += loss.item()
    #         epoch_steps += 1
    #         if i % 2000 == 1999:  # print every 2000 mini-batches
    #             print(
    #                 "[%d, %5d] loss: %.3f"
    #                 % (epoch + 1, i + 1, running_loss / epoch_steps)
    #             )
    #             running_loss = 0.0
        ##########################################

        # Validation loss
        val_loss = 0.0
        val_steps = 0
        total = 0
        correct = 0
        for i, data in enumerate(valloader,0):
            with torch.no_grad():
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)

                preds, _    = net(inputs)
                pred_labels = (F.sigmoid(preds) >= 0.5).type(torch.float32)

                # outputs = net(inputs)
                # _, predicted = torch.max(outputs.data, 1)
                # total += labels.size(0)
                # correct += (predicted == labels).sum().item()

                loss = criterion(pred_labels, labels)
                val_loss += loss.cpu().numpy()
                val_steps += 1

        checkpoint_data = {
            "epoch": epoch,
            "net_state_dict": net.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }
        with tempfile.TemporaryDirectory() as checkpoint_dir:
            data_path = Path(checkpoint_dir) / "data.pkl"
            with open(data_path, "wb") as fp:
                pickle.dump(checkpoint_data, fp)

            checkpoint = tune.Checkpoint.from_directory(checkpoint_dir)
            tune.report(
                {"loss": val_loss / val_steps},
                checkpoint=checkpoint,
            )

    print("Finished Training")

In [31]:
def test_accuracy(model,device = 'cpu'):
    model.eval()
    trainset, testset = load_data()

    testloader = torch.utils.data.DataLoader(
        testset, batch_size=4, shuffle=False, num_workers=2
    )

    all_pred     = torch.empty((0,), dtype = torch.float32).to(device)
    all_labels   = torch.empty((0,), dtype = torch.float32).to(device)
    all_pred_raw = torch.empty((0,), dtype = torch.float32).to(device)

    for data_inputs, data_labels in testloader:
        data_inputs = data_inputs.to(device)
        data_labels = data_labels.to(device)
        preds, _    = model(data_inputs)
        pred_labels = (F.sigmoid(preds) >= 0.5).long()

        all_pred     = torch.cat((all_pred,pred_labels), dim = 0 )
        all_labels   = torch.cat((all_labels,data_labels), dim = 0)
        all_pred_raw = torch.cat([all_pred_raw, preds], dim = 0)

    all_met = (all_metrics(yhat = all_pred.cpu().detach().numpy(), y = all_labels.cpu().detach().numpy(), yhat_raw=all_pred_raw.cpu().detach().numpy()  ))
    return all_met['auc_macro']

In [32]:
num_samples    = 5; # trials
max_num_epochs = 200; # Max epochs
gpus_per_trial = 1
data_dir = os.path.abspath("./data")
load_data(data_dir)
config = {
    "l1": tune.choice([10,15,20]), # Num of Filters
    "l2": tune.choice([3,4,5]),    # Filtern size
    "lr": tune.loguniform(0.0001, 0.1),
    "batch_size": tune.choice([8, 16, 32, 64]),
}
scheduler = ASHAScheduler(
    metric="loss",
    mode="min",
    max_t=max_num_epochs,
    grace_period=1,
    reduction_factor=2,
)
X = np.load('X_train.npy')
total_samples = X.shape[0]
data_points = list(random.choices(population=[0,1,2],k = total_samples))
result = tune.run(
    partial(train_model, data_dir=data_dir,data_points = data_points),
    resources_per_trial={"cpu": 2, "gpu": gpus_per_trial},
    config=config,
    num_samples=num_samples,
    scheduler=scheduler,
)

2025-08-06 05:57:24,815	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


Trial name,loss,should_checkpoint
train_model_39be5_00000,1.11349,True
train_model_39be5_00001,1.01306,True
train_model_39be5_00002,0.968371,True
train_model_39be5_00003,0.969264,True
train_model_39be5_00004,1.04545,True


2025-08-06 06:03:07,573	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/drew/ray_results/train_model_2025-08-06_05-57-24' in 0.0054s.
2025-08-06 06:03:07,580	INFO tune.py:1041 -- Total run time: 342.77 seconds (342.73 seconds for the tuning loop).


In [33]:
best_trial = result.get_best_trial("loss", "min", "last")
print(f"Best trial config: {best_trial.config}")
print(f"Best trial final validation loss: {best_trial.last_result['loss']}")


Best trial config: {'l1': 15, 'l2': 3, 'lr': 0.011123026944109436, 'batch_size': 8}
Best trial final validation loss: 0.9683707262029743


In [34]:
# print(f"Best trial final validation accuracy: {best_trial.last_result['accuracy']}")
best_trained_model = ConvAttnPool(num_of_filters = best_trial.config["l1"], 
                                  kernel_size    = best_trial.config["l2"])

device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")
best_trained_model.to(device)

best_checkpoint = result.get_best_checkpoint(trial=best_trial, mode="max") # , metric="accuracy"
with best_checkpoint.as_directory() as checkpoint_dir:
    print(f'Loading from {checkpoint_dir}')
    data_path = Path(checkpoint_dir) / "data.pkl"
    with open(data_path, "rb") as fp:
        best_checkpoint_data = pickle.load(fp)

    best_trained_model.load_state_dict(best_checkpoint_data["net_state_dict"])
    test_acc = test_accuracy(best_trained_model, device)
    print("Best trial test set accuracy: {}".format(test_acc))

Loading from /home/drew/ray_results/train_model_2025-08-06_05-57-24/train_model_39be5_00002_2_batch_size=8,l1=15,l2=3,lr=0.0111_2025-08-06_05-57-24/checkpoint_000000
Best trial test set accuracy: 0.7545805435413387
